<h1><center>Stock Analysis</center></h1>

Import necessary libraries

In [134]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from alpha_vantage.timeseries import TimeSeries
from alpha_vantage.fundamentaldata import FundamentalData
from alpha_vantage.techindicators import TechIndicators
from alpha_vantage.econindicators import EconIndicators
from dotenv import load_dotenv
import os
import datetime


Import API and retrieve data

In [135]:
load_dotenv()

True

In [136]:
# Imports data
ts = TimeSeries('api_key', output_format='pandas')

df = ts.get_daily('QBTS')

stock = df[0] 

In [137]:
# Converts data into a pandas dataframe
stock = pd.DataFrame(stock)

In [138]:
# View first five rows
stock.head()

,1. open,2. high,3. low,4. close,5. volume
date,,,,,
2025-10-30,32.9100,36.79,32.45,36.11,43960516.0
2025-10-29,32.4750,34.65,32.00,34.26,47219167.0
2025-10-28,35.2954,36.57,31.85,32.00,64548472.0
2025-10-27,33.9900,37.28,33.23,35.04,65054302.0
2025-10-24,33.1950,35.19,32.30,32.65,67660999.0


In [139]:
# View colmns
stock.columns

Index(['1. open', '2. high', '3. low', '4. close', '5. volume'], dtype='object')

In [140]:
# Rename columns
stock.rename(columns={'1. open': 'open', '2. high': 'high', '3. low': 'low', 
'4. close': 'close', '5. volume': 'volume'}, inplace = True)

In [141]:
# view first 5 rows to ensure column names changed.
stock.head()

,open,high,low,close,volume
date,,,,,
2025-10-30,32.9100,36.79,32.45,36.11,43960516.0
2025-10-29,32.4750,34.65,32.00,34.26,47219167.0
2025-10-28,35.2954,36.57,31.85,32.00,64548472.0
2025-10-27,33.9900,37.28,33.23,35.04,65054302.0
2025-10-24,33.1950,35.19,32.30,32.65,67660999.0


In [142]:
stock.tail()

,open,high,low,close,volume
date,,,,,
2025-06-16,15.500,16.79,15.46,16.00,59371520.0
2025-06-13,15.340,15.66,14.84,15.17,41618731.0
2025-06-12,16.590,17.00,15.77,15.88,49982450.0
2025-06-11,17.440,17.71,16.17,16.53,86552097.0
2025-06-10,18.005,18.95,16.86,16.93,64100890.0


In [143]:
stock.shape

(100, 5)

In [144]:
ta = TechIndicators('api_key', output_format = 'pandas')

In [145]:
fd = FundamentalData('api_key' , output_format = 'pandas')

In [146]:
qbts, meta = ta.get_sma('QBTS')

In [147]:
qbts_sma.shape

(0, 7)

In [148]:
qbts_div = fd.get_dividends('QBTS')

In [149]:
qbts_div 


(Empty DataFrame
 Columns: []
 Index: [],
 'QBTS')

In [150]:
qbts_split = fd.get_splits('QBTS')

In [151]:
qbts_split

(Empty DataFrame
 Columns: []
 Index: [],
 'QBTS')

In [152]:
qbts.head()

,SMA
date,
2025-10-30,35.4015
2025-10-29,35.0565
2025-10-28,34.6250
2025-10-27,34.2605
2025-10-24,33.7740


In [153]:
qbts.tail()

,SMA
date,
2022-09-09,8.1410
2022-09-08,8.3950
2022-09-07,8.6570
2022-09-06,8.9740
2022-09-02,9.1925


In [154]:
qbts.shape

(793, 1)

<h1><center>EDA</center><h1>

In [155]:
stock.dtypes

open      float64
high      float64
low       float64
close     float64
volume    float64
dtype: object

In [156]:
stock['pct_ch_clo'] = stock['close'].pct_change() * 100

In [157]:
stock['pct_ch_clo'] = stock['pct_ch_clo'].round(2)

In [158]:
stock.head()

,open,high,low,close,volume,pct_ch_clo
date,,,,,,
2025-10-30,32.9100,36.79,32.45,36.11,43960516.0,NaN
2025-10-29,32.4750,34.65,32.00,34.26,47219167.0,-5.12
2025-10-28,35.2954,36.57,31.85,32.00,64548472.0,-6.60
2025-10-27,33.9900,37.28,33.23,35.04,65054302.0,9.50
2025-10-24,33.1950,35.19,32.30,32.65,67660999.0,-6.82


In [159]:
stock.tail()

,open,high,low,close,volume,pct_ch_clo
date,,,,,,
2025-06-16,15.500,16.79,15.46,16.00,59371520.0,2.89
2025-06-13,15.340,15.66,14.84,15.17,41618731.0,-5.19
2025-06-12,16.590,17.00,15.77,15.88,49982450.0,4.68
2025-06-11,17.440,17.71,16.17,16.53,86552097.0,4.09
2025-06-10,18.005,18.95,16.86,16.93,64100890.0,2.42


In [160]:
stock.shape

(100, 6)

In [161]:
# Function to calculate Sharpe Ratio.
def calculate_sharp_ratio(ret, ann_rfr, periods):
    daily_rfr = ann_rfr/periods
    excess_ret = ret-daily_rfr
    mean_ex_ret = excess_ret.mean()
    std_ex_ret = excess_ret.std()

    if std_ex_ret == 0:
        return np.nan

    sharp_ratio = (mean_ex_ret/std_ex_ret) * np.sqrt(periods)

    return sharp_ratio

In [162]:
# call function to calculate Sharpe Ratio.
ann_rfr = 0.0377
periods = 100
ret = stock['pct_ch_adj']
sharpe = calculate_sharp_ratio(ret, ann_rfr, periods).round(2)
print(f"The Sharpe ratio for QBTS is:\n{sharpe}")

KeyError: 'pct_ch_adj'